# Hydraulic Surrogate Model -- Gradients

This example demonstrates how to compute gradients of different quantities in the context of a hydraulic surrogat model.

In [ ]:
%pip install epyt-control[hydsurrogate] --quiet

In [ ]:
from epyt_control.models import PIGNNModel, device
import torch
import numpy as np

from anytown_helper import create_test_data_anytown

Create and load a pre-trained hydraulic surrogate model for Anytown:

In [ ]:
model = PIGNNModel.from_network("anytown", load_pretrained_model=True)

Create test data:

In [ ]:
heads, reservoir_idx, demands = create_test_data_anytown()  # First axis in 'heads' and 'demands' encode time!

## Simple gradient

We can compute gradients w.r.t. different quantities by calling the [epyt_control.models.PIGNNModel.compute_gradients_from_forward_pass](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.pi_gnn_hydsurrogate.PIGNNModel.compute_gradients_from_forward_pass) function:


In [ ]:
# Compute gradients for the first time step only!
r = model.compute_gradients_from_forward_pass(reservoir_heads=heads[0, reservoir_idx].reshape(1, -1),
                                              demands=demands[0, :].reshape(1, -1))

## Custom loss function

Build a custom loss function and compute gradients w.r.t. the demands through this loss function to the inputs of the surrogate (e.g., demands or other parameters)

In [ ]:
hyd_pred = model.predict(reservoir_heads=heads[0, reservoir_idx].reshape(1, -1),
                         demands=demands[0, :].reshape(1, -1)) 


heads_true = np.random.uniform(0, 1, size=hyd_pred["heads_pred"][0, :].shape).astype(np.float32)  # Generate random ground truth
heads_true = torch.tensor(heads_true, dtype=torch.float32).to(device)
loss = lambda hpred: (hpred - heads_true).sum().view(-1, 1)  # Compare predicted pressure heads to some imaginery ground truth (here, random heads)

In [ ]:
g = model.compute_gradients(reservoir_heads=heads[0, reservoir_idx].reshape(1, -1),
                            demands=demands[0, :].reshape(1, -1),
                            gradient_output="heads",   # We are targeting the predicted pressure heads, not the flow rates!
                            gradient_input="demands",  # We compute gradients w.r.t. the demands
                            output_func=loss)
print(g["grads"].shape)  # Get gradients' shape -- note that they are returned as a torch.Tensor